In [7]:
import asyncio
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# # Ensure UTF-8 output on Windows terminal
# if sys.platform == "win32":
#     sys.stdout.reconfigure(encoding="utf-8", errors="replace")


load_dotenv()

from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage


In [22]:
import os
from claude_agent_sdk import query, ClaudeAgentOptions
# StreamEvent types module se aata hai:
from claude_agent_sdk.types import StreamEvent, AssistantMessage, ResultMessage

options = ClaudeAgentOptions(
   model=os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929"),
   include_partial_messages=True,
) 

async for message in query(prompt="Say hi in 3 words", options=options):
    print("TYPE:", type(message).__name__)
    
    # 1. StreamEvent ke andar .event hota hai
    if isinstance(message, StreamEvent):
        print("  -> STREAM EVENT:", message.event)
        
    # 2. AssistantMessage ke andar .content hota hai
    elif isinstance(message, AssistantMessage):
        print("  -> ASSISTANT CONTENT:", message.content)
        
    # 3. ResultMessage ke andar .result hota hai
    elif isinstance(message, ResultMessage):
        print("  -> RESULT:", message.result)
        
    print("-" * 50)

TYPE: SystemMessage
--------------------------------------------------
TYPE: SystemMessage
--------------------------------------------------
TYPE: StreamEvent
  -> STREAM EVENT: {'type': 'message_start', 'message': {'model': 'claude-sonnet-4-5-20250929', 'id': 'msg_011CgASGce5CTtv2zWkjFUKK', 'type': 'message', 'role': 'assistant', 'content': [], 'container': None, 'stop_reason': None, 'stop_sequence': None, 'stop_details': None, 'usage': {'input_tokens': 10, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 33047, 'cache_creation': {'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_tokens': 3, 'service_tier': 'standard', 'inference_geo': 'not_available', 'speed': 'standard'}, 'diagnostics': None, 'context_management': None}}
--------------------------------------------------
TYPE: StreamEvent
  -> STREAM EVENT: {'type': 'content_block_start', 'index': 0, 'content_block': {'type': 'thinking', 'thinking': '', 'signature': ''}}
--------------------------

### 🧠 Claude Agent SDK: Message Lifecycle & Anatomy Cheat-Sheet

Jab hum `query()` call karte hain (especially with `include_partial_messages=True`), stream se ye 4 primary message types aate hain:

| Message Type | Kahan se import hota hai? | Primary Attribute | Purpose / Asli Kaam |
| :--- | :--- | :--- | :--- |
| **`SystemMessage`** | `claude_agent_sdk.types` | `.subtype`, `.data` | Session start, task initialization aur CLI lifecycle metadata. |
| **`StreamEvent`** | `claude_agent_sdk.types` | `.event` (dict) | Real-time live token chunks (`delta`). Har word aate hi emit hota hai. |
| **`AssistantMessage`** | `claude_agent_sdk.types` | `.content` (list of blocks) | Turn complete hone par aggregated text (`TextBlock`) aur tool requests (`ToolUseBlock`). |
| **`ResultMessage`** | `claude_agent_sdk` / `.types` | `.result`, `.subtype`, `.total_cost_usd` | Final execution summary, total cost ($), turn count, aur completion status. |

---

#### 🔍 StreamEvent Deep-Dive (`message.event` structure):
- `message.event["type"] == "content_block_start"`: Naya block shuru hua.
- `message.event["type"] == "content_block_delta"`: Token chunk aaya!
  - `event["delta"]["type"] == "text_delta"` $\rightarrow$ Actual user-facing word: `event["delta"]["text"]`
  - `event["delta"]["type"] == "thinking_delta"` $\rightarrow$ Model ki internal reasoning: `event["delta"]["thinking"]`
- `message.event["type"] == "content_block_stop"`: Block complete hua.

---

#### ⚠️ 3 Golden Rules jo kabhi nahi bhoolne:
1. **Never do `message.event` directly** bina check kiye ki `isinstance(message, StreamEvent)` hai ya nahi (warna `SystemMessage` crash kar dega).
2. **`ResultMessage` ke paas `.content` nahi hota**, iske paas `.result` hota hai.
3. Live streaming print karte waqt hamesha **`print(token, end="", flush=True)`** use karo taaki console turant flush kare!

In [23]:
import os
from claude_agent_sdk import query, ClaudeAgentOptions

options = ClaudeAgentOptions(
    model=os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-5-20250929"),
    include_partial_messages=True,
)

# Saare message objects ko is list mein save kar lenge:
captured_messages = []

print("Running API call once to capture trajectory...")
async for message in query(prompt="Say hi in 3 words and tell a 1-sentence joke", options=options):
    captured_messages.append(message)

print(f"Captured {len(captured_messages)} messages in memory!")

Running API call once to capture trajectory...
Captured 27 messages in memory!


In [25]:
captured_messages

[SystemMessage(subtype='init', data={'type': 'system', 'subtype': 'init', 'cwd': 'i:\\OpenSource\\Udacity\\we4ai-claude-agents\\day-01-core-loop', 'session_id': 'acef7968-512e-4f60-ad23-2a1f5da65213', 'tools': ['Task', 'Bash', 'CronCreate', 'CronDelete', 'CronList', 'DesignSync', 'Edit', 'EnterWorktree', 'ExitWorktree', 'Glob', 'Grep', 'ListAgents', 'Monitor', 'NotebookEdit', 'PowerShell', 'PushNotification', 'Read', 'ReportFindings', 'ScheduleWakeup', 'SendMessage', 'Skill', 'TaskCreate', 'TaskGet', 'TaskList', 'TaskStop', 'TaskUpdate', 'WebFetch', 'WebSearch', 'Workflow', 'Write'], 'mcp_servers': [], 'model': 'claude-sonnet-4-5-20250929', 'permissionMode': 'default', 'slash_commands': ['drawio-diagram', 'dsa-socratic-sparring', 'make-notes', 'deep-research', 'design', 'design-sync', 'dataviz', 'update-config', 'verify', 'debug', 'code-review', 'simplify', 'batch', 'fewer-permission-prompts', 'doctor', 'loop', 'claude-api', 'workflow-authoring', 'run', 'run-skill-generator', 'plugin-a

In [ ]:
from claude_agent_sdk.types import StreamEvent, AssistantMessage, ResultMessage

print("Replaying captured stream offline:\n")

for message in captured_messages:
    # 1. Live text deltas
    if isinstance(message, StreamEvent):
        event = message.event
        if event.get("type") == "content_block_delta":
            delta = event.get("delta", {})
            if delta.get("type") == "text_delta":
                print(delta.get("text", ""), end="", flush=True)

    # 2. Final result summary
    elif isinstance(message, ResultMessage):
        print("\n\n" + "-" * 40)
        print("📊 Telemetry Summary:")
        print(f"  Status   : {message.subtype}")
        print(f"  Turns    : {message.num_turns}")
        print(f"  Cost     : ${message.total_cost_usd or 0:.4f}")
        print(f"  Duration : {message.duration_ms / 1000:.2f}s")
        print("-" * 40)

Replaying captured stream offline:

Hey there, friend! 

Why don't programmers like nature? It has too many bugs.

----------------------------------------
📊 Telemetry Summary:
  Status   : success
  Turns    : 1
  Cost     : $0.1288
  Duration : 6.18s
----------------------------------------
